<a href="https://colab.research.google.com/github/Shamsfathalla/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shamsfathalla/FlyRank-Starter-Notebooks/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

assert os.path.exists(
    "data/raw/content_refresh_anonymized.csv"
), "starter CSV not found — are you at the repo root?"

print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task type: Scoring.

My lane is a content opportunity scoring problem. The goal is to assign each page a score representing how strongly its observed signals suggest that it may be worth reviewing for a content action such as refresh, expansion, protection, or monitoring. I chose scoring because the main output is a continuous priority value that can be used to rank pages and allocate limited review capacity. The score can combine multiple signals rather than relying on one threshold.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target/proxy: is_declining_label = 1 when trend_direction == "down", otherwise 0.

This is a proxy for identifying pages that may need attention because their observed performance is declining. The label comes from an observed trend field in the starter data rather than from a confirmed business outcome such as a successful refresh. I will therefore treat it as a directional proxy for content opportunity, not as proof that a page needs to be refreshed.

The model can use this proxy to learn a page-level priority score, which can then be used to rank pages for human review.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Target distribution:")
print(df["is_declining_label"].value_counts())

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: ROC-AUC.

I will use ROC-AUC to measure whether the model can rank pages with the declining proxy above pages without the declining proxy. A ROC-AUC of 0.5 represents random ordering, while a value above 0.5 indicates better-than-random separation.

This metric fits the task because the output is intended to prioritize pages for review rather than make a final automatic decision. I will compare the learned approach with a simple rule-based baseline to determine whether ML provides useful ranking signal.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one row = one content page.

Each row represents a single page and contains observed page-level search, engagement, and content signals. These signals can include impressions, sessions, CTR, content age, and trend information.

The model will use page-level signals to produce a priority score for each page. The output can then be sorted so that the content team can review higher-priority pages first.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the unit of analysis as actual rows
page_columns = [
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "content_age_days",
    "trend_direction",
    "is_declining_label"
]

display(df[page_columns].head(10))

,impressions_90d,sessions_90d,ctr,content_age_days,trend_direction,is_declining_label
0,3803,17,0.76,187,down,1
1,15320,9,0.05,445,down,1
2,12581,11,0.09,141,down,1
3,11751,78,0.49,463,stable,0
4,19140,145,0.13,263,down,1
5,3970,5,0.03,147,down,1
6,20,1,0.00,90,down,1
7,1724,28,0.06,445,stable,0
8,32574,68,0.09,90,down,1
9,1240,3,0.16,257,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule could say that a page is high priority when one signal crosses a threshold, such as declining trend or low CTR. The problem is that content opportunity depends on several signals at the same time. A page with high impressions, declining performance, an older content age, and a different position may deserve a different priority from a page that has only one of those characteristics.

ML can learn combinations and relative importance across multiple signals instead of requiring me to manually choose every threshold. This makes it useful for ranking many pages consistently while still keeping the final recommendation as decision support for a human reviewer. I will compare the learned approach with a simple rule-based baseline rather than assuming ML is automatically better.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.